# പാഠം 10 - ഉത്പാദനത്തിൽ AI ഏജന്റുകൾ

ഈ പാഠത്തിൽ Microsoft Agent Framework (Python) ഉപയോഗിച്ച് AI ഏജന്റുകൾക്കുള്ള **ഉത്പാദന മാതൃകകൾ** പഠിക്കും. നമ്മള് കാണുന്നത്:

- **പ്രതീക്ഷണശേഷി** — ഏജന്റ് ഇടപെടലുകളിൽ സമയമാനും ലോഗിംഗിനും കൂട്ടിച്ചേർത്തல்
- **മൂല്യനിർണ്ണയം** — പ്രതികരണ ഗുണനിലവാരത്തിന് സ്കോർ നൽകാൻ മൂല്യനിർണയ ഏജന്റ് ഉപയോഗിക്കൽ
- **ച്ചെലവ് നിയന്ത്രണം** — ടോക്കൺ വിശദീകരണവും മോഡൽ തിരഞ്ഞെടുപ്പും സംബന്ധിച്ച തന്ത്രങ്ങൾ

ഈ രംഗമാണ് **യാത്രാ ഏജന്റ്**; ഉപയോക്താക്കൾക്ക് യാത്രകൾ പദ്ധതിയിടാൻ സഹായിക്കുകയും മേൽനോട്ടവും മൂല്യനിർണയവും ഘടിപ്പിക്കപ്പെടുകയും ചെയ്യുന്നു.


## ക്രമീകരണം


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ഉത്പാദന പരിഗണനകൾ

പ്രോട്ടോടൈപ്പുകളിൽ നിന്നു് AI ഏജന്റുകളെ ഉത്പാദനത്തിലേക്ക് മാറ്റുന്നത് മൂന്നു പ്രധാന ഗുണഭൂതങ്ങളോട് സൂക്ഷ്മ ശ്രദ്ധ ആവശ്യമാണ്:

1. **കാണുന്ന വിധം (Observability)** — ഏജന്റ് എന്ത് ചെയ്യുന്നുവെന്നത്, എത്ര സമയം എടുക്കുന്നു, ഏത് ടൂളുകൾ വിളിക്കുന്നു എന്നതു് നിങ്ങൾക്ക് ദൃശ്യമാകണം. ട്രേസിങ്ങും ലോഗിങ്ങും ഇല്ലാതെ ഉത്പാദന പ്രശ്നങ്ങൾ തിരുത്താൻ כמעט അസാധ്യമാണ്.

2. **പരിശോധന (Evaluation)** — ഓട്ടോമेटഡ് ഗുണനിലവാര പരിശോധനകൾ ഏജന്റിന്റെ മറുപറവുകൾ ശരിയായും പൂർണ്ണമായും സഹായകമായും തുടരുന്നതായി ഉറപ്പാക്കുന്നു. നിർവ്വചിച്ച മാനദണ്ഡങ്ങൾ അനുസരിച്ച് മറുപടികളെ സ്കോർ ചെയ്യാൻ ഒരു വിലയിരുത്തുന്ന ഏജന്റ് ഉപയോഗിക്കാം.

3. **ച്ചെലവ് നിയന്ത്രണം (Cost Management)** — ടോക്കൺ ഉപയോഗം നേരിട്ട് ചെലവിൽ ബാധിക്കുന്നു. പ്രംപ്റ്റ് ഓപ്ടിമൈസേഷൻ, മോഡൽ തിരഞ്ഞെടുപ്പ്, ക്യാഷിംഗ് എന്നിവയുo പോലുള്ള തന്ത്രങ്ങൾ ഗുണനിലവാരം നഷ്ടപ്പെടുത്താതെ ചിലവുകൾ നിയന്ത്രിക്കാൻ സഹായിക്കുന്നു.


## ഒരു അവലോകന ഏജന്റ് നിർമ്മിക്കുക

യാത്രാ ഉപകരണങ്ങൾ നമുക്ക് നിർവചിക്കുകയും, ഏജന്റിന്റെ വിളിയുമായി സമയമെടുക്കലിന്റെ ചുറ്റും പാക്ക് ചെയ്യുകയും ചെയ്യുന്നു, അതുവഴി ഞങ്ങൾ വൈകിപ്പോക്കൽ നിയന്ത്രിക്കാം. പ്രൊഡക്ഷനിൽ നിങ്ങൾ OpenTelemetry അല്ലെങ്കിൽ സമാനമായ ഒരു ട്രേസിംഗ് ബാക്ക്എൻഡുമായി ഇന്റഗ്രേറ്റ് ചെയ്യും.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## മൂല്യനിർണ്ണയ മാതൃകകൾ

സാധാരണ ഉത്പാദന മാതൃക ഒന്നാം ഏജന്റിന്റെ മറുപടികൾ മുൻകൂട്ടി നിശ്ചയിച്ച മാനദണ്ഡങ്ങൾ כגון പൂർണ്ണത, കൃത്യത, സഹായകാരിത്വം എന്നിവയുടെ അടിസ്ഥാനത്തിൽ **മൂല്യനിർണ്ണയകൻ** എന്ന രണ്ടാം ഏജന്റ് ഉപയോഗിക്കുന്നതാണ്.

ഇതുവഴി സാധ്യമാകുന്നത്:
- ഉപയോക്താക്കൾക്ക് മറുപടി ലഭിക്കുന്നതിന് മുമ്പുള്ള ഓട്ടോമാറ്റഡ് ഗുണമേന്മാ ബാരുകൾ
- പ്രോംപ്റ്റുകളോ മോഡലുകളോ മാറ്റുമ്പോൾ പിറകുവരവ് കണ്ടെത്തൽ
- ഏജന്റ് പ്രകടനത്തിന്റെ നിരന്തര നിരീക്ഷണം


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ചെലവ് വിനിയോഗ നയങ്ങൾ

ഉത്പാദന AI ഏജൻറുകൾക്കായി ചെലവുകൾ നിയന്ത്രിക്കുന്നത് നിർണ്ണായകമാണ്. പ്രധാന നയങ്ങൾ ഇവയാണ്:

| നയം | വിവരണം |
|---|---|
| **പ്രോംപ്റ്റ് ഓപ്റ്റിമൈസേഷൻ** | സിസ്റ്റത്തിന്റെ നിർദേശം സംക്ഷിപ്തമാക്കുക. ആവർത്തനപരമായ പാരിസ്ഥിതികം നീക്കംചെയ്യാൻ ഇൻപുട്ട് ടോക്കണുകൾ കുറയ്ക്കുക. |
    "| **മോഡൽ തിരഞ്ഞെടുപ്പ്** | ലഘു, ചെലവു കുറഞ്ഞ മോഡലുകൾ (ഉദാ: GPT-4.1-mini) ക്ലാസിഫിക്കേഷൻ അല്ലെങ്കിൽ എക്‌സ്‌ട്രാക്ഷൻ പോലുള്ള ലളിതമായ ജോലികൾക്കായി ഉപയോഗിക്കുക, ബുദ്ധിമുട്ടുള്ള നിർമ്മിതികൾക്കായി വലുതായ മോഡലുകൾയും സംരക്ഷിക്കുക. |\n",
| **കാഷിംഗ്** |ordi പായ്ക്ക് ഫലങ്ങളും കൂടുതലായി വരുന്ന ചോദനകളും API കയറലുകൾ ആവർത്തിക്കാതെ കാഷ് ചെയ്യുക. |
| **ടോക്കൺ ബജറ്റുകൾ** | പ്രതീക്ഷിക്കാത്ത ദൈർഘ്യമുള്ള മറുപടികൾ തടയാൻ `max_tokens` പരിധികൾ സജ്ജമാക്കുക. |
| **ബാച്ചിങ്** | സാധ്യമായിടത്തോളം പല ഉപയോക്തൃ ചോദനകളെയും ഒരേ API കയറലിൽ കൂട്ടിച്ചേർക്കുക. |

പ്രായോഗികമായി, ഒരു പാളിമാറ്റം സമീപനം നല്ലതാണ്: ലളിതമായ അഭ്യർത്ഥനകളെ വേഗവും കുറഞ്ഞ ചെലവുള്ള മോഡലിലേക്ക് നൽകുകയും സങ്കീർണ്ണമായ ചോദനകൾക്ക് കഴിവുള്ള മോഡലിലേക്ക് മാത്രമേ ഉയർത്തുകയുള്ളൂ.


## സംക്ഷേപം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചതെങ്ങനെ ചെയ്യാമെന്ന്:

1. സൂംവെളിച്ചവും ലോഗ് ചെയ്യലും ഉപയോഗിച്ച് ഏജന്റ് ഇടപെടലുകളിൽ **പരീക്ഷണക്ഷമത ചേർക്കുക**, ട്രേസിംഗ്‌ക്കും മോണിറ്ററിംഗിനും അടിസ്ഥാനം സൃഷ്ടിക്കുക.
2. പൂർണ്ണത, കൃത്യത, സഹായകരത എന്നിവ സ്‌കോർ ചെയ്യുന്ന ഒരു എവാലുവേറ്റർ ഏജന്റ് ഉപയോഗിച്ച് ഏജന്റ് പ്രതികരണങ്ങൾ **സ്വയംമൂല്യനിർണ്ണയം ചെയ്യുക**.
3. പ്രോംപ്റ്റ് ഓപ്റ്റിമൈസേഷൻ, മോഡൽ തിരഞ്ഞെടുപ്പ്, കാഷിംഗ്, ടോക്കൺ ബഡ്ജറ്റുകൾ എന്നിവ മുഖേന **ചെലവുകൾ മാനേജ് ചെയ്യുക**.

ഈ പ്രൊഡക്ഷൻ മാതൃകകൾ നിങ്ങളുടെ AI ഏജന്റുകൾ വിശ്വസനീയവും, അളക്കാവുന്നതുമായും ചെലവുകുറഞ്ഞതുമായും വലുപ്പത്തിൽ ഉറപ്പുവരുത്താൻ സഹായിക്കുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
